# Design matrix — one-camera setup

Single (left) camera variant of `../2_design_matrix.ipynb`.

Differences from the two-camera version:
- Only the left camera is loaded; both paws are read from `leftCamera`.
- Licks come from `get_left_licks` (left camera only) instead of `merge_licks`.
- No `avg_whisker_me` column (needs a second camera).
- One-camera videos can be recorded at **30 or 60 Hz**: the native rate is
  detected per session with `get_frame_rate`, the output grid is fixed at
  **fs = 30 Hz** for cross-session consistency, and an anti-aliasing low-pass
  is applied **only** when a session is downsampled (native > 30 Hz).

In [1]:
import os
import numpy as np
import pickle
import pandas as pd
from brainbox.io.one import SessionLoader
import gc
import concurrent.futures
import gzip

from segmentation_functions import get_left_licks, resample_common_time, interpolate_nans, low_pass, get_frame_rate
from one.api import ONE
one = ONE(mode='remote')

In [4]:
prefix = '/home/ines/repositories/'

In [ ]:
data_path = '/home/ines/repositories/representation_learning_variability/paper-individuality/segmentation/1_camera_setup/temp_data/design_matrices/' # data should be moved to the drive manually
sessions_to_process = list(pd.read_csv('nm_filtered_eids.csv')['eid'])

In [6]:
def process_design_matrix(session):

    file_path = one.eid2path(session)
    if prefix == '/home/ines/repositories/':
        mouse_name = file_path.parts[8]
    else:
        mouse_name = file_path.parts[7]

    """ LOAD VARIABLES """
    sl = SessionLoader(eid=session, one=one)
    sl.load_pose(views=['left'], tracker='lightningPose')
    sl.load_session_data(trials=True, wheel=True, motion_energy=True)

    # Check if all data is available
    if np.sum(sl.data_info['is_loaded']) >= 4:

        # Poses
        poses = sl.pose
        lc_t = np.asarray(poses['leftCamera']['times'])
        # One-camera videos can be recorded at 30 or 60 Hz; detect the native rate
        left_fr = get_frame_rate(lc_t)

        # Motion energy (anti-alias only when downsampling a 60 Hz session to the 30 Hz grid)
        me = sl.motion_energy
        mel_t = lc_t
        if left_fr > 30:
            motion_energy_l = low_pass(interpolate_nans(me['leftCamera']['whiskerMotionEnergy'], left_fr),
                                       cutoff=12, sf=left_fr)
        else:
            motion_energy_l = interpolate_nans(me['leftCamera']['whiskerMotionEnergy'], left_fr)

        # Licks
        features = ['tongue_end_l_x', 'tongue_end_l_y', 'tongue_end_r_x', 'tongue_end_r_y']
        lick_t, licks = get_left_licks(poses, features, common_fs=60)

        # Paws (a single camera sees both paws)
        if left_fr > 30:
            l_paw_x = low_pass(interpolate_nans(poses['leftCamera']['paw_r_x'], left_fr), cutoff=12, sf=left_fr)
            l_paw_y = low_pass(interpolate_nans(poses['leftCamera']['paw_r_y'], left_fr), cutoff=12, sf=left_fr)
            r_paw_x = low_pass(interpolate_nans(poses['leftCamera']['paw_l_x'], left_fr), cutoff=12, sf=left_fr)
            r_paw_y = low_pass(interpolate_nans(poses['leftCamera']['paw_l_y'], left_fr), cutoff=12, sf=left_fr)
        else:
            l_paw_x = interpolate_nans(poses['leftCamera']['paw_r_x'], left_fr)
            l_paw_y = interpolate_nans(poses['leftCamera']['paw_r_y'], left_fr)
            r_paw_x = interpolate_nans(poses['leftCamera']['paw_l_x'], left_fr)
            r_paw_y = interpolate_nans(poses['leftCamera']['paw_l_y'], left_fr)
        l_paw_t = lc_t
        r_paw_t = lc_t

        # Wheel
        wheel = sl.wheel
        wheel_t = np.asarray(wheel['times'], dtype=np.float64)
        wheel_vel = wheel['velocity'].astype(np.float32)

        # Common resampling window
        onset = max(lc_t.min(), wheel_t.min(), lick_t.min())
        offset = min(lc_t.max(), wheel_t.max(), lick_t.max())
        fs = 30
        ref_t = np.arange(onset, offset, 1 / fs, dtype=np.float64)

        # Restrict to time window
        def restrict(t, x):
            mask = (t >= onset) & (t <= offset)
            return t[mask], x[mask]

        mel_t, motion_energy_l = restrict(mel_t, motion_energy_l)
        wheel_t, wheel_vel = restrict(wheel_t, wheel_vel)
        l_paw_t_x, l_paw_x = restrict(l_paw_t, l_paw_x)
        l_paw_t_y, l_paw_y = restrict(l_paw_t, l_paw_y)
        r_paw_t_x, r_paw_x = restrict(r_paw_t, r_paw_x)
        r_paw_t_y, r_paw_y = restrict(r_paw_t, r_paw_y)
        lick_t, licks = restrict(lick_t, licks)

        # Resample
        mel_down, rt = resample_common_time(ref_t, mel_t, motion_energy_l, kind='linear')
        wh_down, _ = resample_common_time(ref_t, wheel_t, wheel_vel, kind='linear')
        lk_down, _ = resample_common_time(ref_t, lick_t, licks, kind='nearest')
        lpx_down, _ = resample_common_time(ref_t, l_paw_t_x, l_paw_x, kind='linear')
        lpy_down, _ = resample_common_time(ref_t, l_paw_t_y, l_paw_y, kind='linear')
        rpx_down, _ = resample_common_time(ref_t, r_paw_t_x, r_paw_x, kind='linear')
        rpy_down, _ = resample_common_time(ref_t, r_paw_t_y, r_paw_y, kind='linear')

        # Create design matrix
        design_matrix = pd.DataFrame({
            'Bin': rt,
            'Lick count': lk_down.astype(np.int8),
            'avg_wheel_vel': wh_down,
            'whisker_me': mel_down,
            'l_paw_x': lpx_down,
            'l_paw_y': lpy_down,
            'r_paw_x': rpx_down,
            'r_paw_y': rpy_down,
        })

        # """ LOAD TRIAL DATA """
        session_trials = sl.trials
        session_start = list(session_trials['goCueTrigger_times'])[0]
        design_matrix = design_matrix.loc[(design_matrix['Bin'] > session_start)]

        """ SAVE DATA """
        # Save unnormalized design matrix
        filename = data_path + "design_matrix_" + str(session) + '_'  + mouse_name
        design_matrix.to_parquet(filename, compression='gzip')

        # Save trials
        filename = data_path + "session_trials_" + str(session) + '_'  + mouse_name
        session_trials.to_parquet(filename, compression='gzip')

        del design_matrix, session_trials, sl
        gc.collect()

    else:
        print('Data missing for session '+session)


def parallel_process_data(sessions, function_name):
    with concurrent.futures.ThreadPoolExecutor() as executor:

        # Process each chunk in parallel
        executor.map(function_name, sessions)

In [7]:
for s, session in enumerate(sessions_to_process):
    process_design_matrix(session)

(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09394/2026-03-05/001/alf/_ibl_leftCamera.lightningPose.pqt: 100%|██████████| 50.0M/50.0M [00:06<00:00, 7.28MB/s]
(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09394/2026-03-05/001/alf/_ibl_leftCamera.times.npy: 100%|██████████| 721k/721k [00:00<00:00, 1.22MB/s]

2026-07-14 15:08:07 INFO     one.py:1288 Loading trials data



(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09394/2026-03-05/001/alf/task_00/_ibl_trials.goCueTrigger_times.npy: 100%|██████████| 5.22k/5.22k [00:00<00:00, 22.5kB/s]
(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09394/2026-03-05/001/alf/task_00/_ibl_trials.stimOffTrigger_times.npy: 100%|██████████| 5.22k/5.22k [00:00<00:00, 22.6kB/s]
(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09394/2026-03-05/001/alf/task_00/_ibl_trials.table.pqt: 100%|██████████| 51.1k/51.1k [00:00<00:00, 164kB/s]
(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09394/2026-03-05/001/alf/task_00/_ibl_trials.quiescencePeriod.npy: 100%|██████████| 5.22k/5.22k [00:00<00:00, 19.8kB/s]
(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09394/2026-03-05/001/alf/task_00/_ibl_trials.stimOnTrigger_times.npy: 100%|██████████| 5.22k/5.22k [00:00

2026-07-14 15:08:10 INFO     one.py:1288 Loading wheel data



(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09394/2026-03-05/001/alf/task_00/_ibl_wheel.position.npy: 100%|██████████| 1.87M/1.87M [00:01<00:00, 988kB/s] 
(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09394/2026-03-05/001/alf/task_00/_ibl_wheel.timestamps.npy: 100%|██████████| 1.87M/1.87M [00:00<00:00, 3.11MB/s]


...

2026-07-14 15:08:30 INFO     one.py:1288 Loading motion_energy data


(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09768/2026-06-18/001/alf/leftCamera.ROIMotionEnergy.npy: 100%|██████████| 1.21M/1.21M [00:00<00:00, 1.52MB/s]

2026-07-14 15:08:32 WARNING  one.py:1292 Could not load motion_energy data.
2026-07-14 15:08:32 INFO     one.py:1288 Loading pupil data



(S3) /home/ines/Downloads/ONE/alyx.internationalbrainlab.org/mainenlab/Subjects/ZFM-09768/2026-06-18/001/alf/_ibl_leftCamera.features.pqt: 100%|██████████| 3.01M/3.01M [00:01<00:00, 2.71MB/s]

2026-07-14 15:08:34 WARNING  one.py:1292 Could not load pupil data.
